Чтение файла с названиями и сайтами

In [ ]:
import pandas as pd
df=pd.read_csv('../Тест на 3 строительные компании_2.csv', delimiter=';')
# print(df)
names=df.iloc[:, 0:4]
sites=df.iloc[:, 4:]
print('names:\n', names)
print('\n\n')
print('sites:\n', sites)

Ключ, папка, модель

In [ ]:
YANDEX_CLOUD_MODEL = ""
YANDEX_CLOUD_API_KEY=""
YANDEX_CLOUD_FOLDER=""

Функции для извлечения полного текста веб-страницы (с лишними элементами)

In [ ]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import time
import os
import json
import re
import sys
from datetime import datetime


def get_full_trafilatura(url, title_need=False):
    """Извлекает оригинальный заголовок и полный текст статьи"""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return None, None
            
        result = trafilatura.extract(
            downloaded,
            include_formatting=True,   # сохраняет абзацы
            include_links=False,
            include_comments=False,
            include_tables=False,
            no_fallback=True,
            output_format="txt"
        )
        
        # trafilatura часто может вытащить и заголовок отдельно
        metadata = trafilatura.extract_metadata(downloaded)
        original_title = metadata.title if metadata and metadata.title else None
        if title_need:
            return original_title
        else:
            return result
        
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")
        return None, None

In [ ]:
from __future__ import annotations
import xml.etree.ElementTree as ET
import pathlib
import time
from typing import List, Dict, Literal, cast
import pandas as pd
from pathlib import Path

from yandex_ai_studio_sdk import AIStudio
from yandex_ai_studio_sdk.search_api import (
    FamilyMode,
    FixTypoMode,
    GroupMode,
    Localization,
    SortMode,
    SortOrder,
)

key="AQVN1TNSln4U0WYxmzmsqRpFYXVX4FzzVpVdNJDS"
id="b1gq04ro9q7t98ku2bce"

FOLDER_ID = id
API_KEY = key

USER_AGENT = (
    "Mozilla/5.0 (Linux; Android 13; Pixel 7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/125.0.6422.112 Mobile Safari/537.36"
)

DATE_FROM = "20250101"
DATE_TO = "20251231"

MAX_SAFE_PAGES = 200  # защита от бесконечного цикла
DELAY_BETWEEN_REQUESTS = 0.5  # задержка для API

In [ ]:
# сбор новостей
def search(company, names, site):
    sdk = AIStudio(
        folder_id=FOLDER_ID,
        auth=API_KEY,
    )

    sdk.setup_default_logging()
    sdk.setup_default_logging("error")

    search = sdk.search_api.web("RU")

    search = search.configure(
        # family_mode=FamilyMode.STRICT,
        fix_typo_mode=FixTypoMode.OFF,
        # group_mode=GroupMode.DEEP,
        group_mode=GroupMode.FLAT,
        localization=Localization.RU,
        sort_mode=SortMode.BY_RELEVANCE,
        sort_order=SortOrder.DESC,
        user_agent=USER_AGENT,
        groups_on_page=100,
    )

    all_results: List[Dict] = []
    name_line=' | '.join(f"'{name}'" for name in names)
    # name_line=' OR '.join(f"'{name}'" for name in names)
    search_query =f"{name_line} site:{site} date:20250101..20251231"
    # search_query =f"'{company}' site:{site} date:20250101..20251231"
    
    print("Поисковый запрос:", search_query)

    for page in range(MAX_SAFE_PAGES):
            print(f"Запрос страницы {page}...")
            try:
                # result_bytes = search.run(search_query, format="xml", page=page) #async
                operation = search.run_deferred(search_query, format="xml", page=page) # sync
                result_bytes = operation.wait(poll_interval=3)
                result_text = result_bytes.decode("utf-8")
                print(result_text)
                # output_filename = (
                #     str(pathlib.Path(__file__).parent)
                #     + "/"
                #     + "page_"
                #     + str(page + 1)
                #     + ".xml"
                # )
                site2=site.split('.')[0]
                timestamp = datetime.now().strftime('%Y-%m-%d')
                output_filename=f'../Альтернативы/alternative_results/{company}_{site2}_2025_page_{page+1}_{timestamp}.xml'
                file = open(output_filename, "a")
                file.write(result_text)
                print(f"Page {page} saved to file {output_filename}")
                file.close()
            except Exception as e:
                print("Ошибка запроса:", e)
                break

            root = ET.fromstring(result_text)
            print(root)
            docs = root.findall(".//doc")
            print(docs)

            if not docs:
                print("Результаты закончились.")
                break

            print(f"Найдено документов: {len(docs)}")
            timestamp = datetime.now().strftime('%Y-%m-%d')
            # Сохраняем XML страницы
            xml_filename = f"../Альтернативы/alternative_results/{company}_{site.split('.')[0]}_2025_page_{page+1}_{timestamp}.xml"
            with open(xml_filename, "w", encoding="utf-8") as f:
                f.write(result_text)

            # Парсим результаты
            for doc in docs:
                url = doc.findtext("url")
                title = doc.findtext("title")
                if len(title)<=10 or "..." in title or title[-1] in ["'", '"', "‘", "“", "«", " "]:
                    title=get_full_trafilatura(url, title_need=True)
                    
                mime = doc.findtext("mime-type")
                date_str=doc.findtext("modtime")
                if date_str:
                    dt = datetime.strptime(date_str, "%Y%m%dT%H%M%S")
                    # Преобразуем в нужный формат
                    formatted_date = dt.strftime("%d.%m.%Y")
                else:
                    formatted_date=""

                if url and mime!="pdf" and mime!="doc" and mime!="docx" and not("pdf" in url.lower()) and not("doc" in url.lower()) and not("xls" in url.lower()) and not("odt" in url.lower()):
                    all_results.append({
                        'NewsFinder': "парсер",
                        "CompanyName": company,
                        "NewsTitle": title,
                        'NewsDate': formatted_date,
                        'NewsSource': site[:-1],
                        "NewsURL": url,
                        "NewsText":"",
                        "CompanyVar": None
                    })

            time.sleep(DELAY_BETWEEN_REQUESTS)

    # Удаляем дубликаты
    unique_results = {item["NewsURL"]: item for item in all_results}.values()
    unique_results = list(unique_results)
    # print(type(unique_results), unique_results)

    print("Всего уникальных статей:", len(unique_results))

    return unique_results

# search('llm', '111', "habr.com/")

In [ ]:
import requests
import time
import base64
import xml.etree.ElementTree as ET
from typing import List, Dict



def search_all_pages_raw(company: str, names: List[str], site: str, max_pages: int = 20) -> List[Dict]:
    all_results: List[Dict] = []
    headers = {
    "Authorization": f"Api-Key {API_KEY}",
    "Content-Type": "application/json"
    }

    # Формируем запрос (как у тебя)
    name_line = ' | '.join(f'"{name}"' for name in names)
    search_query = f'{name_line} site:{site} date:20240101..20241231'

    print(f"Поисковый запрос: {search_query}\n")

    for page in range(max_pages):
        print(f"→ Запрашиваем страницу {page}...")

        payload = {
            "folderId": FOLDER_ID,
            "query": {
                "searchType": "WEB",
                "queryText": search_query,
                "familyMode": "MODERATE",
                "fixTypoMode": "OFF",
                "page": page
            },
            "responseFormat": "XML",
            "groupSpec": {
                "groupMode": "FLAT",      # или "DEEP"
                "groupsOnPage": 100,      # сколько групп (документов) на странице
                "docsInGroup": 1
            }
        }

        # 1. Отправляем отложенный запрос
        try:
            resp = requests.post(
                "https://searchapi.api.cloud.yandex.net/v2/web/searchAsync",
                headers=headers,
                json=payload,
                timeout=20
            )
            resp.raise_for_status()
            operation = resp.json()
            operation_id = operation["id"]
        except Exception as e:
            print(f"Ошибка отправки страницы {page}: {e}")
            break

        # 2. Ждём завершения операции
        url_status = f"https://operation.api.cloud.yandex.net/operations/{operation_id}"
        max_wait = 180
        interval = 3

        for _ in range(max_wait // interval + 1):
            try:
                status_resp = requests.get(url_status, headers=headers, timeout=15)
                status_data = status_resp.json()

                if status_data.get("done"):
                    if "error" in status_data:
                        print(f"Ошибка выполнения: {status_data['error']}")
                    else:
                        # Декодируем результат
                        raw_base64 = status_data["response"]["rawData"]
                        xml_text = base64.b64decode(raw_base64).decode("utf-8")

                        # Парсим XML
                        root = ET.fromstring(xml_text)
                        docs = root.findall(".//doc")

                        if not docs:
                            print(f"Страница {page}: результатов больше нет.")
                            return all_results  # выходим полностью

                        print(f"Страница {page}: найдено документов — {len(docs)}")

                        for doc in docs:
                            url = doc.findtext("url")
                            title = doc.findtext("title") or ""
                            mime = doc.findtext("mime-type")
                            date_str = doc.findtext("modtime")

                            formatted_date = ""
                            if date_str:
                                try:
                                    dt = datetime.strptime(date_str, "%Y%m%dT%H%M%S")
                                    formatted_date = dt.strftime("%d.%m.%Y")
                                except:
                                    pass

                            # Фильтруем ненужные форматы
                            if (url and 
                                mime not in ("pdf", "doc", "docx") and 
                                not any(ext in url.lower() for ext in [".pdf", ".doc", ".docx", ".xls", ".odt"])):

                                # Если заголовок плохой — можно дополнить через trafilatura
                                if len(title.strip()) <= 15 or "..." in title:
                                    title = get_full_trafilatura(url, title_need=True) or title

                                all_results.append({
                                    'NewsFinder': "парсер",
                                    "CompanyName": company,
                                    "NewsTitle": title.strip(),
                                    'NewsDate': formatted_date,
                                    'NewsSource': site.rstrip('/'),
                                    "NewsURL": url,
                                    "NewsText": "",
                                    "CompanyVar": None
                                })
                    break

                time.sleep(interval)
            except Exception as e:
                print(f"Ошибка проверки статуса: {e}")
                break
        else:
            print(f"Страница {page}: превышено время ожидания")

        time.sleep(1.5)  # задержка между страницами

    # Удаляем дубликаты по URL
    unique_results = {item["NewsURL"]: item for item in all_results}.values()
    print(f"\nВсего уникальных статей после всех страниц: {len(unique_results)}")
    
    return list(unique_results)

In [ ]:
# формируем таблицу с полным текстом новостей
import json
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

def formed_table(company, name, site):
    client = OpenAI(
        api_key=YANDEX_CLOUD_API_KEY,
        base_url="https://ai.api.cloud.yandex.net/v1",
        project=YANDEX_CLOUD_FOLDER
    )
    articles=search(company, name, site)
    # articles=search_all_pages_raw(company, name, site)
    try:
        ad=articles[0]["NewsSource"]
        pd.DataFrame(articles).to_csv(f"Urls_{company}_{name}_{ad}.csv")
    except: pass
    articles_text = []
    urls=[]

    for i in articles:
        i["NewsText"]=get_full_trafilatura(i["NewsURL"])

    print(articles)
    return articles

# formed_table('kindle paperwhite', 'kindle paperwhite', 'habr.com/')

In [ ]:
# сбор новостей
for i in range(len(df)):
    # формирование новостей по официальному названию
    company_names=names.iloc[i, :]
    company_sites=sites.iloc[i, :]
    # name_line=' | '.join(f'"{name}"' for name in list(company_names))
    # print(name_line)

    for site in company_sites:
        news_full=formed_table(company_names.iloc[0], list(company_names), site)

        timestamp = datetime.now().strftime('%Y-%m-%d-%H-%M')
        site2=site.split(".")[0]
        csv_filename = f"../Альтернативы/alternative_results/news_test_yandex_{company_names.iloc[0]}_{site2}_{timestamp}.csv"
        print(csv_filename)

        df = pd.DataFrame(news_full)
        print(df)
        try:
            df['NewsTitle'] = df['NewsTitle'].str.replace(r'"', "'", regex=True)
            df['NewsTitle'] = '"' + df['NewsTitle'] + '"'

            df['NewsText'] = df['NewsText'].str.replace(r'[\n\r\t]+', ' ', regex=True)
            # df['NewsTitle'] = df['NewsTitle'].str.replace(r'\s+', ' ', regex=True).str.strip()
            df['NewsText'] = df['NewsText'].str.replace(r'"', "'", regex=True)
            df['NewsText'] = '"' + df['NewsText'] + '"'
        except: pass
        
        df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
